# Retentioneering — API showcase

This notebook exercises every public `Eventstream` method on the bundled e-commerce dataset: here is the input, here is the call, here is the output, and here is what each argument means. No storytelling — for a narrative analytics walkthrough see `ecom_demo.ipynb`.

Two intended uses:

- **Users** — skim top to bottom to learn what each tool does and how its arguments work. Every section links to the corresponding page of the [documentation](https://retentioneering.com/docs/).
- **Developers** — run top to bottom as a smoke test: no cell should raise, and every widget should render.

Widget cells only use **data parameters** (the ones that change the computed result). Display parameters (`height`, `sidebar_open`, ...) are deliberately left at their defaults — see the per-widget documentation pages for those.

## Setup

The dataset: six months of a consumer-electronics store — 600 users, ~3600 sessions, ~31000 events, 19 event types, plus four segment columns. The [schema](https://retentioneering.com/docs/eventstream#schema) declares column roles. `path_cols` defines what a *path* is (see [Key concepts](https://retentioneering.com/docs/eventstream#key-concepts)): the first column is the default grain (here — the whole user journey), and every tool accepts a `path_col` override to switch to session grain.

In [1]:
import pandas as pd

from retentioneering import Eventstream
from retentioneering.datasets.ecom import load_ecom

df = load_ecom()

stream = Eventstream(
    df,
    schema={
        "path_cols": ["user_id", "session_id"],
        "segment_cols": [
            "platform",
            "acquisition_channel",
            "user_cohort",
            "user_lifecycle",
        ],
    },
)

# A user with a purchase — used below to show before/after transformations
USER = df.loc[df["event"] == "purchase", "user_id"].iloc[0]

print(f"rows: {len(df):,}   users: {df['user_id'].nunique()}   demo user: {USER}")
df.head()

rows: 31,106   users: 600   demo user: user_0000


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning


## 1. Eventstream basics

Docs: [Eventstream](https://retentioneering.com/docs/eventstream).

`Eventstream` wraps the dataframe and exposes everything else as methods.
Basic introspection first.

In [2]:
# The data back as a plain DataFrame.
# exclude_start_end=True (default) hides the synthetic path_start/path_end rows.
stream.to_dataframe().head()

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5


In [3]:
# How many times each event occurs
counts = stream.get_event_counts()
dict(sorted(counts.items(), key=lambda kv: -kv[1]))

{'product_view': 6851,
 'catalog': 6521,
 'search': 4011,
 'home': 2762,
 'filter_results': 2610,
 'add_to_cart': 1346,
 'cart': 1204,
 'shipping_details': 958,
 'wishlist_add': 812,
 'promo_page': 796,
 'payment_details': 709,
 'account_page': 659,
 'purchase': 470,
 'review_page': 406,
 'support_chat': 277,
 'error_page': 276,
 'compare': 275,
 'payment_error': 119,
 'checkout_bug': 44}

In [4]:
# Declared segment columns and their values
stream.get_segment_values()

{'platform': ['desktop', 'mobile', 'tablet'],
 'acquisition_channel': ['direct',
  'email',
  'organic',
  'paid_search',
  'social'],
 'user_cohort': ['2024-01', '2024-02', '2024-03', '2024-04'],
 'user_lifecycle': ['loyal', 'new', 'returning']}

In [5]:
# is_empty()  — cheap emptiness check
# fingerprint — stable content hash of the data shape / event distribution
# equals()    — compare two eventstreams
print("is_empty:", stream.is_empty())
print("fingerprint:", stream.fingerprint)
print("equals(itself + no-op filter):", stream.equals(stream.filter_events()))

is_empty: False
fingerprint: a58473b1f75f08686b7601d768baba44
equals(itself + no-op filter): True


## 2. Data processors

Docs: [Data processors](https://retentioneering.com/docs/data-processors).

Every processor returns a **new** `Eventstream` — the original is never
modified, so calls chain freely. Each cell below starts from the same
`stream` and renders the resulting dataframe, so you can see exactly how the
rows were transformed.

In [6]:
def peek(s, user=None, n=10):
    """First rows of a resulting eventstream, optionally for one user."""
    out = s.to_dataframe()
    if user is not None:
        out = out[out["user_id"] == user]
    return out.head(n)

### filter_events — keep rows

Docs: [Filter Events](https://retentioneering.com/docs/data-processors/filter-events).

Four mutually exclusive modes, one argument each: `keep`, `drop`, `func`,
`sql`. `keep` takes a `{column: values}` mapping; multiple columns combine
with **AND** (a row must match every entry).

In [7]:
res = stream.filter_events(
    keep={"event": ["add_to_cart", "purchase"], "platform": ["mobile"]}
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0004,sess_000028,add_to_cart,2024-03-30 14:07:48,mobile,social,2024-02,returning,raw,1,16
1,user_0005,sess_000032,add_to_cart,2024-06-04 15:14:04,mobile,paid_search,2024-04,returning,raw,1,4
2,user_0005,sess_000034,add_to_cart,2024-06-11 14:44:12,mobile,paid_search,2024-04,returning,raw,1,42
3,user_0010,sess_000058,add_to_cart,2024-04-21 20:33:35,mobile,paid_search,2024-03,returning,raw,1,12
4,user_0010,sess_000059,add_to_cart,2024-06-06 08:53:24,mobile,paid_search,2024-03,loyal,raw,1,35
5,user_0010,sess_000059,add_to_cart,2024-06-06 09:09:27,mobile,paid_search,2024-03,loyal,raw,1,45
6,user_0010,sess_000061,purchase,2024-06-12 12:39:08,mobile,paid_search,2024-03,loyal,raw,1,58
7,user_0010,sess_000061,add_to_cart,2024-06-12 12:41:20,mobile,paid_search,2024-03,loyal,raw,1,59
8,user_0010,sess_000062,add_to_cart,2024-06-14 14:00:54,mobile,paid_search,2024-03,loyal,raw,1,67
9,user_0010,sess_000062,purchase,2024-06-14 14:04:50,mobile,paid_search,2024-03,loyal,raw,1,70


### filter_events — drop rows

Same `{column: values}` format, but removes matches. Multiple columns combine
with **OR** (a row is removed if it matches *any* entry) — the exact
complement of `keep`.

In [8]:
res = stream.filter_events(
    drop={"event": ["support_chat", "error_page"], "platform": ["tablet"]}
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### filter_events — Python predicate

`func` receives the raw DataFrame and returns a boolean mask; `True` rows are
kept.

In [9]:
res = stream.filter_events(func=lambda d: d["timestamp"].dt.month == 3)  # March only
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### filter_events — SQL

`sql` takes a DuckDB SELECT that reads from the fixed alias `eventstream` and
returns all original columns.

In [10]:
res = stream.filter_events(
    sql="SELECT * FROM eventstream WHERE event NOT LIKE '%_error'"
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### filter_paths — keep whole paths by metric conditions

Docs: [Filter Paths](https://retentioneering.com/docs/data-processors/filter-paths),
[Path Metrics](https://retentioneering.com/docs/path-metrics) (the full metric list).

Unlike `filter_events` (row-level), `filter_paths` keeps or drops **entire
paths**. The condition is a tree of comparisons over per-path metrics;
operators are `=` (or `==`), `!=`, `>`, `<`, `>=`, `<=`, combined with
`and` / `or` / `not` nodes.

In [11]:
# Paths that contain at least one purchase
res = stream.filter_paths(
    {
        "op": ">",
        "metric": "event_count",
        "value": 0,
        "metric_args": {"events": "purchase"},
    }
)
print("users kept:", res.to_dataframe()["user_id"].nunique())
peek(res)

users kept: 328


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


In [12]:
# A top-level list is shorthand for AND: long paths that follow cart -> ... -> purchase
res = stream.filter_paths(
    [
        {"op": ">", "metric": "length", "value": 20},
        {
            "op": "=",
            "metric": "matches_pattern",
            "value": True,
            "metric_args": {"pattern": "add_to_cart->.*->purchase"},
        },
    ]
)
print("users kept:", res.to_dataframe()["user_id"].nunique())
peek(res)

users kept: 293


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### add_events — from source events

Docs: [Add Events](https://retentioneering.com/docs/data-processors/add-events).

Inserts synthetic events; original rows are kept. In `source_events` mode one
event is inserted per path, at the timestamp of the **first** occurrence of
any source event.

In [13]:
res = stream.add_events("first_cart_touch", source_events=["add_to_cart", "cart"])
peek(res, user=USER)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,first_cart_touch,2024-03-13 10:23:32,desktop,email,2024-01,returning,synthetic,2,4
5,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
6,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
7,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
8,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
9,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9


### add_events — from SQL

Every row returned by the query (alias `eventstream`, same columns) becomes a
new synthetic event named `name`.

In [14]:
res = stream.add_events(
    "payment_issue", sql="SELECT * FROM eventstream WHERE event = 'payment_error'"
)
res.to_dataframe().query("event == 'payment_issue'").head()

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
376,user_0006,sess_000043,payment_issue,2024-05-23 13:29:22,desktop,paid_search,2024-01,loyal,synthetic,2,94
507,user_0009,sess_000056,payment_issue,2024-05-27 13:01:39,desktop,organic,2024-01,loyal,synthetic,2,34
627,user_0011,sess_000065,payment_issue,2024-05-21 16:56:48,mobile,paid_search,2024-04,new,synthetic,2,42
686,user_0013,sess_000071,payment_issue,2024-05-26 09:44:30,tablet,organic,2024-04,returning,synthetic,2,15
891,user_0017,sess_000095,payment_issue,2024-05-29 16:15:54,desktop,email,2024-01,loyal,synthetic,2,52


### add_events — churn marker

`churn` inserts an event after a period of inactivity: `inactivity_days` is
the gap threshold; optional `active_events` restricts what counts as
activity.

In [15]:
res = stream.add_events(
    "churned",
    churn={"inactivity_days": 30, "active_events": ["purchase", "add_to_cart"]},
)
res.to_dataframe().query("event == 'churned'").head()

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
17,user_0000,sess_000002,churned,2024-03-23 09:53:12,desktop,email,2024-01,returning,synthetic,2,17
26,user_0000,sess_000005,churned,2024-04-30 19:37:15,desktop,email,2024-01,loyal,synthetic,2,25
95,user_0001,sess_000015,churned,2024-05-22 20:56:56,desktop,organic,2024-04,returning,synthetic,2,48
113,user_0002,sess_000018,churned,2024-03-24 11:59:48,desktop,paid_search,2024-03,new,synthetic,2,9
152,user_0003,sess_000021,churned,2024-03-23 21:43:47,desktop,paid_search,2024-02,returning,synthetic,2,25


### add_segment — CASE-WHEN rules

Docs: [Add Segment](https://retentioneering.com/docs/data-processors/add-segment). A *segment column*
names a segmentation; each value is one segment (static or dynamic — see
[Key concepts](https://retentioneering.com/docs/eventstream#key-concepts)).

`rules` is a list of `[column, op, value, label]` conditions plus a final
`[else_label]` entry.

In [16]:
res = stream.add_segment(
    "channel_type",
    rules=[
        ["acquisition_channel", "in", "('paid_search', 'social', 'email')", "paid"],
        ["organic_or_direct"],
    ],
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index,channel_type
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1,paid
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2,paid
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3,paid
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4,paid
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5,paid
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6,paid
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7,paid
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8,paid
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9,paid
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10,paid


### add_segment — Python function

`func` gets the raw DataFrame and returns one label per row. Because labels
are assigned per **row**, segments can be *dynamic* — here the value changes
along the path depending on the event's weekday.

In [17]:
res = stream.add_segment(
    "day_type",
    func=lambda d: (d["timestamp"].dt.dayofweek >= 5).map(
        {True: "weekend", False: "weekday"}
    ),
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index,day_type
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1,weekday
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2,weekday
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3,weekday
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4,weekday
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5,weekday
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6,weekday
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7,weekday
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8,weekday
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9,weekday
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10,weekday


### add_segment — SQL

The query (alias `eventstream`) must return exactly one column with one label
per row, in the original row order. Window functions are supported.

In [18]:
res = stream.add_segment(
    "session_size",
    sql="""
        SELECT CASE WHEN COUNT(*) OVER (PARTITION BY session_id) >= 10
                    THEN 'long_session' ELSE 'short_session' END
        FROM eventstream
    """,
)
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index,session_size
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1,long_session
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2,long_session
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3,long_session
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4,long_session
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5,long_session
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6,long_session
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7,long_session
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8,long_session
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9,long_session
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10,long_session


### add_segment — funnel stage

`funnel_events` assigns each path the name of the **last funnel step reached
in sequence** (or `out_of_funnel`). Handy for drop-off analysis: diff the
paths that stopped at step N against those that reached N+1.

In [19]:
res = stream.add_segment(
    "funnel_stage", funnel_events=["add_to_cart", "payment_details", "purchase"]
)
print(res.to_dataframe().groupby("funnel_stage", observed=True)["user_id"].nunique())
peek(res)

funnel_stage
add_to_cart        303
out_of_funnel       78
payment_details    144
purchase            75
Name: user_id, dtype: int64


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index,funnel_stage
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1,payment_details
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2,payment_details
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3,payment_details
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4,payment_details
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5,payment_details
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6,payment_details
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7,payment_details
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8,payment_details
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9,payment_details
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10,payment_details


### add_clusters — ML segment from behavioral features

Docs: [Add Clusters](https://retentioneering.com/docs/data-processors/add-clusters).

Computes per-path metrics (`features`), scales them (`minmax` / `standard` /
`None`), clusters (`kmeans` needs `n_clusters`; `hdbscan` takes
`min_cluster_size` / `cluster_selection_epsilon`), and broadcasts the label
to every row of the path. `nmf_components` optionally reduces features with
NMF first. For interactive exploration use the
[Cluster Analysis widget](https://retentioneering.com/docs/widgets/cluster-analysis) instead.

In [20]:
clustered = stream.add_clusters(
    name="behavior_cluster",
    features=[
        {"metric": "length"},
        {
            "metric": "event_count",
            "metric_args": {"events": ["purchase", "add_to_cart", "search"]},
        },
    ],
    method="kmeans",
    n_clusters=4,
    scaler="minmax",
)
print(
    clustered.to_dataframe()
    .groupby("behavior_cluster", observed=True)["user_id"]
    .nunique()
)
peek(clustered)

behavior_cluster
cluster_0    159
cluster_1     85
cluster_2    152
cluster_3    204
Name: user_id, dtype: int64


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index,behavior_cluster
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1,cluster_1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2,cluster_1
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3,cluster_1
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4,cluster_1
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5,cluster_1
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6,cluster_1
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7,cluster_1
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8,cluster_1
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9,cluster_1
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10,cluster_1


### drop_segment

Removes a segment column added earlier (or declared in the schema).

In [21]:
res = clustered.drop_segment("behavior_cluster")
print("segments:", res.schema.segment_cols)
peek(res)

segments: ['platform', 'acquisition_channel', 'user_cohort', 'user_lifecycle']


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### collapse_events — consecutive repeats

Docs: [Collapse Events](https://retentioneering.com/docs/data-processors/collapse-events).

Four mutually exclusive modes. `consecutive=True` merges runs of the same
event (`product_view -> product_view -> product_view` becomes one row); pass
a list of event names to collapse only those.

In [22]:
res = stream.collapse_events(consecutive=True)
print(f"rows: {len(stream.to_dataframe()):,} -> {len(res.to_dataframe()):,}")
peek(res, user=USER)

rows: 31,106 -> 27,174


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,index,subindex
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,2,1
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,3,1
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,4,1
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,5,1
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,6,1
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,collapsed,7,1
7,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,10,1
8,user_0000,sess_000001,search,2024-03-13 10:47:30,desktop,email,2024-01,returning,raw,11,1
9,user_0000,sess_000002,product_view,2024-03-23 09:44:28,desktop,email,2024-01,returning,raw,12,1


### collapse_events — event groups

`event_groups` merges a set of related events into one representative event
(labelled `default`), taking the group's first timestamp.

In [23]:
res = stream.collapse_events(
    event_groups=[
        {
            "events": ["payment_details", "shipping_details", "review_page"],
            "default": "checkout",
        },
    ]
)
peek(res, user=USER)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,index,subindex
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,2,1
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,3,1
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,4,1
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,5,1
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,6,1
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,7,1
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,8,1
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,9,1
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,10,1


### collapse_events — group by a column

`group_col` collapses each run of consecutive rows sharing the same column
value into one event **named after that value**. Here we first label rows as
`browse`/`action` and then collapse the path into alternating stages.

In [24]:
staged = stream.add_segment(
    "screen_type",
    rules=[
        [
            "event",
            "in",
            "('home', 'catalog', 'search', 'filter_results', 'product_view', 'compare', 'promo_page')",
            "browse",
        ],
        ["action"],
    ],
)
res = staged.collapse_events(group_col="screen_type")
peek(res, user=USER)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,screen_type,event_type,index,subindex
0,user_0000,sess_000001,browse,2024-03-13 10:21:14,desktop,email,2024-01,returning,browse,collapsed,1,1
1,user_0000,sess_000001,action,2024-03-13 10:23:32,desktop,email,2024-01,returning,action,collapsed,4,1
2,user_0000,sess_000001,browse,2024-03-13 10:27:06,desktop,email,2024-01,returning,browse,collapsed,5,1
3,user_0000,sess_000002,action,2024-03-23 09:44:51,desktop,email,2024-01,returning,action,collapsed,13,1
4,user_0000,sess_000003,browse,2024-04-21 17:51:34,desktop,email,2024-01,loyal,browse,collapsed,18,1
5,user_0000,sess_000004,action,2024-04-27 16:45:48,desktop,email,2024-01,loyal,action,collapsed,20,1
6,user_0000,sess_000004,browse,2024-04-27 16:46:21,desktop,email,2024-01,loyal,browse,collapsed,21,1
7,user_0000,sess_000004,action,2024-04-27 16:52:36,desktop,email,2024-01,loyal,action,collapsed,22,1
8,user_0000,sess_000005,browse,2024-04-30 19:33:58,desktop,email,2024-01,loyal,browse,collapsed,24,1
9,user_0000,sess_000005,action,2024-04-30 19:37:15,desktop,email,2024-01,loyal,action,collapsed,25,1


### collapse_events — one event per session

`session_id_col` + `session_type_col` collapse every session into a single
event named after the session's type. Here each session becomes
`buying_session` or `browsing_session`, so a user's path reads as a sequence
of sessions.

In [25]:
typed = stream.add_segment(
    "session_kind",
    sql="""
        SELECT CASE WHEN SUM(CASE WHEN event = 'purchase' THEN 1 ELSE 0 END)
                         OVER (PARTITION BY session_id) > 0
                    THEN 'buying_session' ELSE 'browsing_session' END
        FROM eventstream
    """,
)
res = typed.collapse_events(
    session_id_col="session_id", session_type_col="session_kind"
)
peek(res, user=USER)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,session_kind,event_type,index,subindex
0,user_0000,sess_000001,browsing_session,2024-03-13 10:21:14,desktop,email,2024-01,returning,browsing_session,collapsed,1,1
1,user_0000,sess_000002,buying_session,2024-03-23 09:44:28,desktop,email,2024-01,returning,buying_session,collapsed,12,1
2,user_0000,sess_000003,browsing_session,2024-04-21 17:51:34,desktop,email,2024-01,loyal,browsing_session,collapsed,18,1
3,user_0000,sess_000004,browsing_session,2024-04-27 16:45:48,desktop,email,2024-01,loyal,browsing_session,collapsed,20,1
4,user_0000,sess_000005,browsing_session,2024-04-30 19:33:30,desktop,email,2024-01,loyal,browsing_session,collapsed,23,1
5,user_0000,sess_000006,browsing_session,2024-05-07 13:10:22,desktop,email,2024-01,loyal,browsing_session,collapsed,26,1
6,user_0000,sess_000007,buying_session,2024-06-12 16:23:40,desktop,email,2024-01,loyal,buying_session,collapsed,34,1
7,user_0000,sess_000008,buying_session,2024-06-18 19:12:28,desktop,email,2024-01,loyal,buying_session,collapsed,40,1


### rename_events / drop_events / edit_events

Docs: [Rename Events](https://retentioneering.com/docs/data-processors/rename-events),
[Drop Events](https://retentioneering.com/docs/data-processors/drop-events),
[Edit Events](https://retentioneering.com/docs/data-processors/edit-events).

`rename_events` maps old names to new ones; `drop_events` removes events by
name (unknown names raise); `edit_events` does both in one pass.

In [26]:
res = stream.rename_events({"wishlist_add": "add_to_wishlist", "cart": "cart_view"})
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


In [27]:
res = stream.drop_events(["support_chat", "error_page"])
print("events left:", res.to_dataframe()["event"].astype(str).nunique())
peek(res)

events left: 17


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


In [28]:
res = stream.edit_events(rename={"promo_page": "promo"}, delete=["account_page"])
peek(res)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
1,user_0000,sess_000001,promo,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
2,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
3,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
4,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
5,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
6,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
7,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
8,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
9,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10


### sample_paths — by count

Docs: [Sample Paths](https://retentioneering.com/docs/data-processors/sample-paths).

Randomly samples whole paths (all their events survive). Mirrors
`pandas.DataFrame.sample`: `n` for a count, `frac` for a share, and
`random_state` for reproducibility.

In [29]:
res = stream.sample_paths(n=100, random_state=42)
print("users:", res.to_dataframe()["user_id"].nunique())
peek(res)

users: 100


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0005,sess_000031,catalog,2024-05-19 12:33:07,mobile,paid_search,2024-04,returning,raw,1,1
1,user_0005,sess_000032,catalog,2024-06-04 15:11:48,mobile,paid_search,2024-04,returning,raw,1,2
2,user_0005,sess_000032,home,2024-06-04 15:13:52,mobile,paid_search,2024-04,returning,raw,1,3
3,user_0005,sess_000032,add_to_cart,2024-06-04 15:14:04,mobile,paid_search,2024-04,returning,raw,1,4
4,user_0005,sess_000032,product_view,2024-06-04 15:16:33,mobile,paid_search,2024-04,returning,raw,1,5
5,user_0005,sess_000032,filter_results,2024-06-04 15:19:18,mobile,paid_search,2024-04,returning,raw,1,6
6,user_0005,sess_000032,catalog,2024-06-04 15:21:09,mobile,paid_search,2024-04,returning,raw,1,7
7,user_0005,sess_000032,catalog,2024-06-04 15:21:20,mobile,paid_search,2024-04,returning,raw,1,8
8,user_0005,sess_000032,product_view,2024-06-04 15:27:40,mobile,paid_search,2024-04,returning,raw,1,9
9,user_0005,sess_000033,search,2024-06-09 13:30:46,mobile,paid_search,2024-04,returning,raw,1,10


### sample_paths — by fraction

In [30]:
res = stream.sample_paths(frac=0.2, random_state=42)
print("users:", res.to_dataframe()["user_id"].nunique())
peek(res)

users: 120


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0002,sess_000017,home,2024-03-20 19:38:38,desktop,paid_search,2024-03,new,raw,1,1
1,user_0002,sess_000017,catalog,2024-03-20 19:40:21,desktop,paid_search,2024-03,new,raw,1,2
2,user_0002,sess_000017,search,2024-03-20 19:42:28,desktop,paid_search,2024-03,new,raw,1,3
3,user_0002,sess_000017,product_view,2024-03-20 19:45:32,desktop,paid_search,2024-03,new,raw,1,4
4,user_0002,sess_000017,add_to_cart,2024-03-20 19:52:25,desktop,paid_search,2024-03,new,raw,1,5
5,user_0002,sess_000018,home,2024-03-24 11:57:46,desktop,paid_search,2024-03,new,raw,1,6
6,user_0002,sess_000018,catalog,2024-03-24 11:58:48,desktop,paid_search,2024-03,new,raw,1,7
7,user_0002,sess_000018,catalog,2024-03-24 11:59:10,desktop,paid_search,2024-03,new,raw,1,8
8,user_0002,sess_000018,add_to_cart,2024-03-24 11:59:48,desktop,paid_search,2024-03,new,raw,1,9
9,user_0002,sess_000018,catalog,2024-03-24 12:00:30,desktop,paid_search,2024-03,new,raw,1,10


### split_sessions — inactivity timeout

Docs: [Split Sessions](https://retentioneering.com/docs/data-processors/split-sessions).

Adds `session_id` / `session_index` columns (names configurable via
`session_id_col` / `session_index_col`). Three boundary modes: `timeout`,
`separator`, `start_event`+`end_event`; `timeout` can be combined with either.

`timeout` takes a duration **string with an explicit unit** (`"30m"`, `"1h"`,
`"1800s"`) or a `pandas.Timedelta` — bare numbers are rejected to avoid unit
ambiguity. The ecom dataset already carries sessions, so these cells work on
a flattened copy without them.

In [31]:
flat = Eventstream(df[["user_id", "event", "timestamp"]].copy())

res = flat.split_sessions(timeout="30m")
peek(res, user=USER, n=12)

,user_id,event,timestamp,event_type,index,subindex,session_index,session_id
0,user_0000,home,2024-03-13 10:21:14,raw,1,1,1,user_0000_1
1,user_0000,promo_page,2024-03-13 10:21:28,raw,2,1,1,user_0000_1
2,user_0000,product_view,2024-03-13 10:22:12,raw,3,1,1,user_0000_1
3,user_0000,add_to_cart,2024-03-13 10:23:32,raw,4,1,1,user_0000_1
4,user_0000,product_view,2024-03-13 10:27:06,raw,5,1,1,user_0000_1
5,user_0000,catalog,2024-03-13 10:27:40,raw,6,1,1,user_0000_1
6,user_0000,product_view,2024-03-13 10:28:06,raw,7,1,1,user_0000_1
7,user_0000,product_view,2024-03-13 10:29:13,raw,8,1,1,user_0000_1
8,user_0000,product_view,2024-03-13 10:37:44,raw,9,1,1,user_0000_1
9,user_0000,catalog,2024-03-13 10:45:29,raw,10,1,1,user_0000_1


### split_sessions — separator event

The separator event starts a new session and belongs to it — natural when
the log has an explicit "app opened / landing" event; here `home` plays that
role.

In [32]:
res = flat.split_sessions(separator="home")
peek(res, user=USER, n=12)

,user_id,event,timestamp,event_type,index,subindex,session_index,session_id
0,user_0000,promo_page,2024-03-13 10:21:28,raw,2,1,1,user_0000_1
1,user_0000,product_view,2024-03-13 10:22:12,raw,3,1,1,user_0000_1
2,user_0000,add_to_cart,2024-03-13 10:23:32,raw,4,1,1,user_0000_1
3,user_0000,product_view,2024-03-13 10:27:06,raw,5,1,1,user_0000_1
4,user_0000,catalog,2024-03-13 10:27:40,raw,6,1,1,user_0000_1
5,user_0000,product_view,2024-03-13 10:28:06,raw,7,1,1,user_0000_1
6,user_0000,product_view,2024-03-13 10:29:13,raw,8,1,1,user_0000_1
7,user_0000,product_view,2024-03-13 10:37:44,raw,9,1,1,user_0000_1
8,user_0000,catalog,2024-03-13 10:45:29,raw,10,1,1,user_0000_1
9,user_0000,search,2024-03-13 10:47:30,raw,11,1,1,user_0000_1


### split_sessions — start/end events

Sessions span from a `start_event` to an `end_event`; rows outside any such
span get an empty session. Best when the log has explicit markers — here we
use `home -> purchase` spans purely to show the mechanics.

In [33]:
res = flat.split_sessions(start_event="home", end_event="purchase")
peek(res, user=USER, n=12)

,user_id,event,timestamp,event_type,index,subindex,session_index,session_id
0,user_0000,promo_page,2024-03-13 10:21:28,raw,2,1,1,user_0000_1
1,user_0000,product_view,2024-03-13 10:22:12,raw,3,1,1,user_0000_1
2,user_0000,add_to_cart,2024-03-13 10:23:32,raw,4,1,1,user_0000_1
3,user_0000,product_view,2024-03-13 10:27:06,raw,5,1,1,user_0000_1
4,user_0000,catalog,2024-03-13 10:27:40,raw,6,1,1,user_0000_1
5,user_0000,product_view,2024-03-13 10:28:06,raw,7,1,1,user_0000_1
6,user_0000,product_view,2024-03-13 10:29:13,raw,8,1,1,user_0000_1
7,user_0000,product_view,2024-03-13 10:37:44,raw,9,1,1,user_0000_1
8,user_0000,catalog,2024-03-13 10:45:29,raw,10,1,1,user_0000_1
9,user_0000,search,2024-03-13 10:47:30,raw,11,1,1,user_0000_1


### truncate_paths — window between two anchor events

Docs: [Truncate Paths](https://retentioneering.com/docs/data-processors/truncate-paths).

Keeps, per path, the window from the first `start_event` to the first
`end_event` after it (inclusive); paths without both anchors in order are
removed. The reserved names `path_start` / `path_end` work as anchors too.

In [34]:
res = stream.truncate_paths(start_event="add_to_cart", end_event="purchase")
print("users kept:", res.to_dataframe()["user_id"].nunique())
peek(res, user=USER)

users kept: 297


,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
1,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
2,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
3,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7
4,user_0000,sess_000001,product_view,2024-03-13 10:29:13,desktop,email,2024-01,returning,raw,1,8
5,user_0000,sess_000001,product_view,2024-03-13 10:37:44,desktop,email,2024-01,returning,raw,1,9
6,user_0000,sess_000001,catalog,2024-03-13 10:45:29,desktop,email,2024-01,returning,raw,1,10
7,user_0000,sess_000001,search,2024-03-13 10:47:30,desktop,email,2024-01,returning,raw,1,11
8,user_0000,sess_000002,product_view,2024-03-23 09:44:28,desktop,email,2024-01,returning,raw,1,12
9,user_0000,sess_000002,cart,2024-03-23 09:44:51,desktop,email,2024-01,returning,raw,1,13


### to_daily_states — lifecycle resampling

Docs: [To Daily States](https://retentioneering.com/docs/data-processors/to-daily-states).

Converts the stream into one row per path per calendar day, labelled with an
engagement state: active days are `new` / `current` / `reactivated` /
`resurrected`, inactive days are `at_risk_wau` / `at_risk_mau` / `dormant`.
`active_events` restricts what counts as activity; `max_dormant_days` caps
how long rows are generated after the last event.

In [35]:
daily = stream.to_daily_states(
    active_events=["catalog", "add_to_cart", "purchase"], max_dormant_days=14
)
print(daily.to_dataframe()["event"].value_counts())
peek(daily, user=USER, n=12)

event
at_risk_mau    24494
at_risk_wau    17523
dormant        11341
reactivated     1239
current          792
new              600
resurrected      461
Name: count, dtype: int64


,user_id,timestamp,session_id,event,event_type,subindex,index,platform,acquisition_channel,user_cohort,user_lifecycle
0,user_0000,2024-03-13,<NA>,new,collapsed,1,1,desktop,email,2024-01,returning
1,user_0000,2024-03-14,<NA>,at_risk_wau,collapsed,1,2,desktop,email,2024-01,returning
2,user_0000,2024-03-15,<NA>,at_risk_wau,collapsed,1,3,desktop,email,2024-01,returning
3,user_0000,2024-03-16,<NA>,at_risk_wau,collapsed,1,4,desktop,email,2024-01,returning
4,user_0000,2024-03-17,<NA>,at_risk_wau,collapsed,1,5,desktop,email,2024-01,returning
5,user_0000,2024-03-18,<NA>,at_risk_wau,collapsed,1,6,desktop,email,2024-01,returning
6,user_0000,2024-03-19,<NA>,at_risk_wau,collapsed,1,7,desktop,email,2024-01,returning
7,user_0000,2024-03-20,<NA>,at_risk_wau,collapsed,1,8,desktop,email,2024-01,returning
8,user_0000,2024-03-21,<NA>,at_risk_mau,collapsed,1,9,desktop,email,2024-01,returning
9,user_0000,2024-03-22,<NA>,at_risk_mau,collapsed,1,10,desktop,email,2024-01,returning


### add_start_end_events

Docs: [Add Start End Events](https://retentioneering.com/docs/data-processors/add-start-end-events).

Prepends `path_start` and appends `path_end` to each path (idempotent). You
rarely call it yourself — the widgets insert these markers on the fly, each
respecting its own `path_col`.

In [36]:
res = stream.add_start_end_events()
out = res.to_dataframe(exclude_start_end=False)  # keep the markers visible
out[out["user_id"] == USER].head(8)

,user_id,session_id,event,timestamp,platform,acquisition_channel,user_cohort,user_lifecycle,event_type,subindex,index
0,user_0000,sess_000001,path_start,2024-03-13 10:21:14,desktop,email,2024-01,returning,path_start,0,1
1,user_0000,sess_000001,home,2024-03-13 10:21:14,desktop,email,2024-01,returning,raw,1,1
2,user_0000,sess_000001,promo_page,2024-03-13 10:21:28,desktop,email,2024-01,returning,raw,1,2
3,user_0000,sess_000001,product_view,2024-03-13 10:22:12,desktop,email,2024-01,returning,raw,1,3
4,user_0000,sess_000001,add_to_cart,2024-03-13 10:23:32,desktop,email,2024-01,returning,raw,1,4
5,user_0000,sess_000001,product_view,2024-03-13 10:27:06,desktop,email,2024-01,returning,raw,1,5
6,user_0000,sess_000001,catalog,2024-03-13 10:27:40,desktop,email,2024-01,returning,raw,1,6
7,user_0000,sess_000001,product_view,2024-03-13 10:28:06,desktop,email,2024-01,returning,raw,1,7


### urls_to_events — structured events from raw URLs

Docs: [URLs to Events](https://retentioneering.com/docs/data-processors/urls-to-events).

Turns a URL column into event names using a path tree. Node keys:
`aggregate_children` collapses deeper pages into `<path>/<slug>`; `exclude`
drops the subtree; `name` sets a custom label/slug. `strip_host` /
`strip_query` / `strip_locale` (all default `True`) normalize the URL first.
The ecom dataset has no URLs, so this cell uses a tiny synthetic log.

In [37]:
url_df = pd.DataFrame(
    {
        "user_id": ["u1"] * 6,
        "event": ["page_view"] * 6,
        "timestamp": pd.date_range("2024-01-01 10:00", periods=6, freq="1min"),
        "url": [
            "https://shop.example.com/en/catalog/phones/iphone-15",
            "https://shop.example.com/en/catalog/laptops",
            "https://shop.example.com/en/cart",
            "https://shop.example.com/en/checkout/payment?step=2",
            "https://shop.example.com/en/admin/debug",
            "https://shop.example.com/en/checkout/confirm",
        ],
    }
)
url_stream = Eventstream(url_df, schema={"custom_cols": ["url"]})

res = url_stream.urls_to_events(
    column="url",
    nodes=[
        {"path": "/catalog", "aggregate_children": True},
        {"path": "/catalog/phones", "name": "phones"},
        {"path": "/checkout", "aggregate_children": True},
        {"path": "/admin", "exclude": True},
    ],
)
res.to_dataframe()

,user_id,event,timestamp,url,event_type,subindex,index
0,u1,page_view:/catalog/phones,2024-01-01 10:00:00,https://shop.example.com/en/catalog/phones/iph...,raw,1,1
1,u1,page_view:/catalog/sub-page,2024-01-01 10:01:00,https://shop.example.com/en/catalog/laptops,raw,1,2
2,u1,page_view:/cart,2024-01-01 10:02:00,https://shop.example.com/en/cart,raw,1,3
3,u1,page_view:/checkout/sub-page,2024-01-01 10:03:00,https://shop.example.com/en/checkout/payment?s...,raw,1,4
5,u1,page_view:/checkout/sub-page,2024-01-01 10:05:00,https://shop.example.com/en/checkout/confirm,raw,1,6


## 3. Path metrics

Docs: [Path Metrics](https://retentioneering.com/docs/path-metrics).

One registry of per-path scalar metrics powers `filter_paths`,
`add_clusters`, Segment Overview, and Cluster Analysis — and is available
raw via `get_metrics()` (e.g. as ML features). The cell below uses **every
metric** once.

In [38]:
metrics = stream.get_metrics(
    [
        {"metric": "length"},  # events per path
        {"metric": "duration"},  # seconds, first to last event
        {"metric": "active_days"},  # distinct active calendar days
        {
            "metric": "event_count",
            "metric_args": {"events": "purchase"},
        },  # occurrences of an event
        {"metric": "has_event", "metric_args": {"events": "purchase"}},  # 0/1 presence
        {
            "metric": "time_between",
            "metric_args": {
                "start_event": "path_start",  # seconds between two anchors
                "end_event": "purchase",
            },
        },
        {"metric": "first_event_time"},  # unix ts of the first event
        {
            "metric": "matches_pattern",
            "metric_args": {"pattern": "add_to_cart->.*->purchase"},
        },
        {
            "metric": "in_segment",
            "metric_args": {
                "segment_name": "user_lifecycle",
                "segment_value": "loyal",
                "mode": "any",
            },
        },
    ]
)
metrics.head(8)

,length,duration,active_days,event_count_purchase,has_event_purchase,time_from_path_start_to_purchase,first_event_time,matches_pattern_add_to_cart->.*->purchase,in_segment_user_lifecycle_loyal_any
user_id,,,,,,,,,
user_0000,45,8413336.0,8,3,1,862318.0,1.710325e+09,True,1
user_0001,56,6395080.0,8,0,0,NaN,1.712934e+09,False,0
user_0002,22,8085862.0,3,1,1,8085862.0,1.710964e+09,True,1
user_0003,75,7106955.0,7,2,1,561197.0,1.710669e+09,True,1
user_0004,40,9208787.0,4,1,1,1087.0,1.709846e+09,True,1
user_0005,44,1996039.0,4,0,0,NaN,1.716122e+09,False,0
user_0006,96,11076817.0,9,0,0,NaN,1.705395e+09,False,1
user_0007,17,4419700.0,3,0,0,NaN,1.714586e+09,False,0


### get_metric_distribution

Distribution of one metric for a segment value vs. its complement (or vs. a
second value) — the same computation behind Segment Overview's clickable
cells. Returns histogram bins/counts, KDE, and the Wasserstein distance.

In [39]:
dist = stream.get_metric_distribution(
    segment_col="platform",
    segment_value="mobile",
    metric={"metric": "length"},
    complement=True,  # mobile vs. everyone else
)
print("keys:", list(dist.keys()))
print("distance (mobile vs rest):", round(dist["distance"], 3))

keys: ['distribution_1', 'distribution_2', 'distance', 'log_scale']
distance (mobile vs rest): 2.223


## 4. Widgets

Docs: [Widgets](https://retentioneering.com/docs/widgets).

Common behavior for all widgets:

- every data parameter is also editable live in the **sidebar** — no need to
  re-run the cell;
- every widget has a headless `*_data()` twin returning raw data
  ([headless mode](https://retentioneering.com/docs/widgets#headless-mode));
- every widget supports [diff mode](https://retentioneering.com/docs/widgets#diff-mode):
  `diff=[segment_col, value1, value2]` overlays two groups; the reserved
  `"<REST>"` as `value2` means "everyone else".

### Transition Graph — basic

Docs: [Transition Graph](https://retentioneering.com/docs/widgets/transition-graph).

Nodes are events, edges are transitions. `edge_weight` picks the edge value:
`proba_out` (default) / `proba_in` — transition probabilities out of the
source / into the target; `count`; `unique_paths`; `share_of_total`;
`avg_per_path`; `time_median` / `time_q95` (seconds).

In [40]:
stream.transition_graph()

### Transition Graph — diff mode

Two segments in one graph: edge colors show where `value2` behaves
differently from `value1`. Try `diff=["platform", "mobile", "<REST>"]` for
one-vs-everyone-else.

In [41]:
stream.transition_graph(diff=["platform", "mobile", "desktop"])

In [42]:
# Headless twin: the transition matrix as an events x events DataFrame
tm = stream.transition_graph_data(edge_weight="count")
tm.iloc[:8, :8]

next_event,path_start,account_page,add_to_cart,cart,catalog,checkout_bug,compare,error_page
event,,,,,,,,
path_start,0,7,2,1,161,0,0,0
account_page,0,11,28,18,155,0,4,4
add_to_cart,0,23,19,223,162,8,10,17
cart,0,19,159,34,122,9,14,18
catalog,0,136,212,124,1239,5,50,54
checkout_bug,0,0,10,9,5,1,0,1
compare,0,6,18,11,56,0,1,0
error_page,0,0,10,10,56,0,1,5


### Step Matrix — basic

Docs: [Step Matrix](https://retentioneering.com/docs/widgets/step-matrix).

A heatmap of `P(event | step)`: each column is an ordinal step of the path
and sums to 1. `max_steps` controls how many steps are computed.

In [43]:
stream.step_matrix(max_steps=12)

### Step Matrix — path_pattern

`path_pattern` is a `->`-separated sequence of anchor events where `.*`
matches any run of events. Paths are filtered to those matching the pattern,
and each anchor gets its own matrix block with steps counted relative to it
(negative = before the anchor). A serrated edge means paths continue beyond
the visible range.

In [44]:
stream.step_matrix(path_pattern=".*->add_to_cart->.*->purchase")

### Step Matrix — diff mode

In diff mode each column sums to 0: cells show the share difference
`value2 − value1`. Here: new users vs. everyone else (`"<REST>"`).

In [45]:
stream.step_matrix(diff=["user_lifecycle", "new", "<REST>"])

In [46]:
# Headless twin (shared with Step Sankey): one DataFrame per anchor block
blocks = stream.step_matrix_data(max_steps=8)
blocks[0].round(3)

step,0,1,2,3,4,5,6,7,8
event,,,,,,,,,
path_start,1.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
account_page,0.0,0.012,0.025,0.022,0.020,0.020,0.032,0.022,0.012
add_to_cart,0.0,0.003,0.012,0.025,0.028,0.033,0.045,0.032,0.035
cart,0.0,0.002,0.008,0.007,0.022,0.025,0.027,0.030,0.032
catalog,0.0,0.268,0.248,0.255,0.242,0.240,0.192,0.237,0.225
checkout_bug,0.0,0.000,0.002,0.000,0.000,0.000,0.000,0.000,0.002
compare,0.0,0.000,0.005,0.005,0.008,0.013,0.013,0.007,0.010
error_page,0.0,0.000,0.003,0.005,0.005,0.010,0.012,0.007,0.015
filter_results,0.0,0.028,0.087,0.098,0.105,0.088,0.092,0.103,0.092


### Step Sankey — basic

Docs: [Step Sankey](https://retentioneering.com/docs/widgets/step-sankey).

The same per-step data as Step Matrix, drawn as a flow diagram — use
whichever makes the pattern easier to spot. Same `max_steps`,
`path_pattern`, and `diff` arguments.

In [47]:
stream.step_sankey(max_steps=8)

### Step Sankey — path_pattern

In [48]:
stream.step_sankey(path_pattern=".*->purchase")

### Step Sankey — diff mode

In [49]:
stream.step_sankey(diff=["acquisition_channel", "paid_search", "organic"])

### Funnel — basic

Docs: [Funnel](https://retentioneering.com/docs/widgets/funnel).

Sequential semantics: a path counts at step N only if that step's event
occurs **after** all previous steps already happened.

In [50]:
stream.funnel(steps=["catalog", "add_to_cart", "cart", "purchase"])

### Funnel — diff mode

In [51]:
stream.funnel(
    steps=["catalog", "add_to_cart", "cart", "purchase"],
    diff=["platform", "mobile", "desktop"],
)

In [52]:
# Headless twin: conversion numbers as a dict
stream.funnel_data(steps=["catalog", "add_to_cart", "cart", "purchase"])

{'steps': [{'step': 'catalog', 'unique_paths': 600, 'conversion_rate': 1.0},
  {'step': 'add_to_cart',
   'unique_paths': 511,
   'conversion_rate': 0.8516666666666667},
  {'step': 'cart', 'unique_paths': 431, 'conversion_rate': 0.7183333333333334},
  {'step': 'purchase', 'unique_paths': 225, 'conversion_rate': 0.375}]}

### Segment Overview

Docs: [Segment Overview](https://retentioneering.com/docs/widgets/segment-overview),
[Path Metrics](https://retentioneering.com/docs/path-metrics).

Rows are metrics, columns are segment values; `segment_size` and
`segment_share` are always added. Each metric carries an `agg` — `mean`,
`median`, `q5`/`q25`/`q75`/`q95`, or `complement_distance` (Wasserstein
distance between this segment's distribution and everyone else's: higher =
more distinctive). **Click** a cell to see the metric's distribution for
that segment; **shift-click** a second cell in the same row to compare two.

The config below uses every metric from the registry once.

In [53]:
stream.segment_overview(
    segment_col="user_lifecycle",
    metrics=[
        {"metric": "length", "agg": "mean"},
        {"metric": "duration", "agg": "median"},
        {"metric": "active_days", "agg": "mean"},
        {"metric": "event_count", "metric_args": {"events": "purchase"}, "agg": "mean"},
        {"metric": "has_event", "metric_args": {"events": "purchase"}, "agg": "mean"},
        {
            "metric": "time_between",
            "metric_args": {"start_event": "path_start", "end_event": "purchase"},
            "agg": "median",
        },
        {"metric": "first_event_time", "agg": "q5"},
        {
            "metric": "matches_pattern",
            "metric_args": {"pattern": "search->.*->purchase"},
            "agg": "mean",
        },
        {
            "metric": "in_segment",
            "metric_args": {
                "segment_name": "platform",
                "segment_value": "mobile",
                "mode": "event_share",
                "threshold": 0.5,
            },
            "agg": "complement_distance",
        },
    ],
)

In [54]:
# Headless twin: the same table as a DataFrame
stream.segment_overview_data(
    segment_col="platform",
    metrics=[
        {"metric": "length", "agg": "mean"},
        {"metric": "has_event", "metric_args": {"events": "purchase"}, "agg": "mean"},
    ],
).round(3)

platform,desktop,mobile,tablet
metric,,,
segment_size,392.000,297.000,107.000
segment_share,0.492,0.373,0.134
length_mean,41.189,38.226,33.710
has_event_purchase_mean,0.482,0.330,0.458


### Cluster Analysis

Docs: [Cluster Analysis](https://retentioneering.com/docs/widgets/cluster-analysis).

Clusters paths by behavioral `features` and shows a Segment Overview-style heatmap of `overview_metrics` for the resulting clusters. Both arguments take configs from the same [Path Metrics](https://retentioneering.com/docs/path-metrics) registry — features are the clustering *input*, overview metrics *describe* the output.

**How the optimal-split search works.** `n_clusters` accepts a single int, a list, or a range string like `"3-6"`. For a list/range the widget runs a grid search: for every candidate *k* it fits KMeans and computes the **silhouette score** (how much closer each path is to its own cluster than to the nearest other one, from −1 to 1; on a sample of up to 2 000 paths for speed). The *k* with the best score wins, and the whole score curve is drawn in the widget so you can see how close the runner-ups were. The same grid mechanism works for `nmf_components` (NMF decomposition of the feature matrix, toggleable in the sidebar) and, with the `hdbscan` method, for `min_cluster_size` / `cluster_selection_epsilon`.

In [55]:
stream.cluster_analysis(
    features=[
        {"metric": "length"},
        {"metric": "duration"},
        {"metric": "active_days"},
        {
            "metric": "event_count",
            "metric_args": {
                "events": ["purchase", "add_to_cart", "search", "product_view"]
            },
        },
        {"metric": "has_event", "metric_args": {"events": "checkout_bug"}},
    ],
    method="kmeans",
    scaler="minmax",
    n_clusters="3-6",
    overview_metrics=[
        {"metric": "length", "agg": "mean"},
        {"metric": "has_event", "metric_args": {"events": "purchase"}, "agg": "mean"},
        {
            "metric": "time_between",
            "metric_args": {"start_event": "path_start", "end_event": "purchase"},
            "agg": "median",
        },
        {
            "metric": "matches_pattern",
            "metric_args": {"pattern": "checkout_bug->.*->purchase"},
            "agg": "mean",
        },
        {"metric": "first_event_time", "agg": "median"},
        {
            "metric": "in_segment",
            "metric_args": {
                "segment_name": "user_lifecycle",
                "segment_value": "loyal",
                "mode": "any",
            },
            "agg": "mean",
        },
    ],
)

In [56]:
# Headless twin: the silhouette grid is returned as data, so you can inspect
# exactly what the search saw
result = stream.cluster_analysis_data(
    features=[
        {"metric": "length"},
        {
            "metric": "event_count",
            "metric_args": {"events": ["purchase", "add_to_cart"]},
        },
    ],
    n_clusters=[3, 4, 5, 6],
)
for params, score in zip(
    result["silhouette"]["params"], result["silhouette"]["silhouette"]
):
    print(params, "-> silhouette", round(score, 3))
result["overview_df"].round(3)

{'n_clusters': 3} -> silhouette 0.35
{'n_clusters': 4} -> silhouette 0.292
{'n_clusters': 5} -> silhouette 0.346
{'n_clusters': 6} -> silhouette 0.36


__cluster__,cluster_0,cluster_1,cluster_2,cluster_3,cluster_4,cluster_5
metric,,,,,,
segment_size,151.000,62.000,119.000,39.000,93.000,136.000
segment_share,0.252,0.103,0.198,0.065,0.155,0.227


## Where to go next

- [Documentation](https://retentioneering.com/docs/) — guides and the full API reference.
- `ecom_demo.ipynb` — a narrative walkthrough of the same dataset: stories,
  anomalies, and how to investigate them with these tools.
- [MCP server](https://retentioneering.com/docs/mcp) — `retentioneering.mcp.serve(stream)` exposes the
  eventstream to Claude or any MCP client for agent-driven analysis.